In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 比較用データセット（Fusionモデルと同じものを使用）
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"Single_ConvNeXt_model_{TARGET_EPOCH}epoch"


# --- 1. ConvNeXt パーツ定義 ---
class LayerNorm2d(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.normalized_shape = (normalized_shape, )

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        x = self.weight[:, None, None] * x + self.bias[:, None, None]
        return x

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path=0.):
        super().__init__()
        # 1. Depthwise Conv 7x7
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim) 
        self.norm = LayerNorm2d(dim, eps=1e-6)
        
        # 2. Pointwise Conv (Expansion)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        
        # 3. Pointwise Conv (Projection)
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True)

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = self.norm(x)
        
        # Conv2d (NCHW) -> Linear (NHWC) に変換して計算
        x = x.permute(0, 2, 3, 1) 
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2) # 元に戻す

        x = input + x # 残差結合
        return x

# --- 2. ConvNeXt (Custom Tiny / Nano) 単独モデル ---
class ConvNeXtClassifier(nn.Module):
    def __init__(self, num_classes):
        super(ConvNeXtClassifier, self).__init__()
        
        # 設定: 軽量化版 (Custom Nano相当)
        # Fusionモデルで使用したものと同じ規模感にします
        self.depths = [2, 2, 6, 2]
        self.dims = [64, 128, 256, 512]
        
        # Stem: 画像サイズを1/4にダウンサンプリング
        self.downsample_layers = nn.ModuleList() 
        stem = nn.Sequential(
            nn.Conv2d(3, self.dims[0], kernel_size=4, stride=4),
            LayerNorm2d(self.dims[0], eps=1e-6)
        )
        self.downsample_layers.append(stem)
        
        # Downsample layers for stages 2-4
        for i in range(3):
            downsample_layer = nn.Sequential(
                LayerNorm2d(self.dims[i], eps=1e-6),
                nn.Conv2d(self.dims[i], self.dims[i+1], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(downsample_layer)

        # Stages
        self.stages = nn.ModuleList() 
        for i in range(4):
            stage = nn.Sequential(
                *[ConvNeXtBlock(dim=self.dims[i]) for _ in range(self.depths[i])]
            )
            self.stages.append(stage)
        
        # Classifier Head (Global Average Pooling -> Linear)
        # Fusionと異なり、直接クラス分類を行います
        self.norm = nn.LayerNorm(self.dims[-1], eps=1e-6) # Final Norm
        self.head = nn.Linear(self.dims[-1], num_classes)

    def forward(self, x):
        # 特徴抽出
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        
        # 分類ヘッド (GAP + Linear)
        # x shape: (B, C, H, W)
        x = x.mean([-2, -1]) # Global Average Pooling -> (B, C)
        x = self.norm(x)
        x = self.head(x)
        return x

# --- 3. Datasetクラス (以前と同じものを流用) ---
# ※ ハンドクラフト特徴量も返しますが、学習ループで無視します
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)
    plt.legend()
    plt.grid(True)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 5. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    # Dataset & DataLoader
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (ConvNeXt単独)
    model = ConvNeXtClassifier(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        # DataLoaderからは (image, hc_feat, label) が返るが、hc_featは無視する
        for images, _, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            # hc_feat は使わない
            
            optimizer.zero_grad()
            outputs = model(images) # 画像のみ入力
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        # 検証
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, _, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # ベストモデル保存
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_convnext_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.2306 | Val Acc: 0.3333
  --> Best Model Saved (Acc: 0.3333)
Epoch 2/100 | Loss: 1.1388 | Val Acc: 0.3333
Epoch 3/100 | Loss: 1.1257 | Val Acc: 0.3333
Epoch 4/100 | Loss: 1.1145 | Val Acc: 0.3333
Epoch 5/100 | Loss: 1.1059 | Val Acc: 0.3403
  --> Best Model Saved (Acc: 0.3403)
Learning curve saved to Single_ConvNeXt_model_100epoch/learning_curve.png
Epoch 6/100 | Loss: 1.0993 | Val Acc: 0.3333
Epoch 7/100 | Loss: 1.1066 | Val Acc: 0.3333
Epoch 8/100 | Loss: 1.1030 | Val Acc: 0.3333
Epoch 9/100 | Loss: 1.0991 | Val Acc: 0.3333
Epoch 10/100 | Loss: 1.0865 | Val Acc: 0.4097
  --> Best Model Saved (Acc: 0.4097)
Learning curve saved to Single_ConvNeXt_model_100epoch/learning_curve.png
Epoch 11/100 | Loss: 1.0671 | Val Acc: 0.4375
  --> Best Model Saved (Acc: 0.4375)
Epoch 12/100 | Loss: 1.0513 | Val Acc: 0.3889
Epoch 13/100 | Loss: 1.0327 | Val Acc: 0.4583
  --

ReaNet18

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 比較用データセット（共通）
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"Standard_ResNet18_model_{TARGET_EPOCH}epoch"


# --- 1. 標準 ResNet BasicBlock (CBAMなし) ---
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        
        # 1つ目の畳み込み
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # 2つ目の畳み込み
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # ★ ここにあったCBAMを削除

        # ショートカット接続
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        
        # ★ CBAM適用なし
        
        out += identity
        out = self.relu(out)

        return out

# --- 2. 標準 ResNet-18 Classifier ---
class ResNet18Classifier(nn.Module):
    def __init__(self, num_classes=3):
        super(ResNet18Classifier, self).__init__()
        
        # Stem (初期層)
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Layers [2, 2, 2, 2]
        self.layer1 = self._make_layer(64,  blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)
        self.layer4 = self._make_layer(512, blocks=2, stride=2)
        
        # Classifier Head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        # 最初のブロック（stride適用用）
        layers.append(BasicBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        # 残りのブロック
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        # 特徴抽出
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # 分類
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)
    plt.legend()
    plt.grid(True)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 5. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (標準ResNet18)
    model = ResNet18Classifier(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, _, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            # hc_featは無視
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, _, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_std_resnet18_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.1630 | Val Acc: 0.4167
  --> Best Model Saved (Acc: 0.4167)
Epoch 2/100 | Loss: 1.0672 | Val Acc: 0.4028


KeyboardInterrupt: 